# **<span style="color:red"> -Scrapping Data From Naukri.com Website </span>**

<br>
<br>
<br>

### **Code**

In [2]:
import os
import time
import logging
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()



# Set up the Selenium WebDriver using Service and ChromeDriverManager
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# Initialize empty lists to store scraped data
CompanyName = []
JobTitle = []
Experience = []
Skills = []
Links = []

# -------------------------------
# 2. Web Scraping Function
# -------------------------------

def scrape_job_listings(page_number, job_type, url_pattern):
    job_url = url_pattern.format(page_number=page_number)
    
    logger.info(f"Fetching URL: {job_url}")
    
    # Open the webpage
    driver.get(job_url)
    
    # Wait for the page to load
    time.sleep(5)
    
    # Get the page source
    page_content = driver.page_source
    
    # Parse the HTML with BeautifulSoup
    soup = BeautifulSoup(page_content, "html.parser")
    
    # Find all job listing boxes on the page
    boxes = soup.find_all('div', class_="cust-job-tuple layout-wrapper lay-2 sjw__tuple")
    
    logger.info(f"Found {len(boxes)} {job_type} job listings on page {page_number}")
    
    # Extract data from each job listing
    for box in boxes:
        # Extract Company Name
        company = box.find('a', class_='comp-name')
        company_name = company.text.strip() if company else 'N/A'
        CompanyName.append(company_name)
        
        # Extract Job Title
        job = box.find('a', class_='title')
        job_title = job.text.strip() if job else 'N/A'
        JobTitle.append(job_title)
        
        # Extract Experience
        experience = box.find('span', class_='expwdth')
        experience_text = experience.text.strip() if experience else 'N/A'
        Experience.append(experience_text)
        
        # Extract Skills
        skills_list = box.find('ul', class_='tags-gt')
        if skills_list:
            skills = [skill.text.strip() for skill in skills_list.find_all('li')]
            skills_text = ', '.join(skills)
            Skills.append(skills_text)
        else:
            Skills.append('N/A')
        
        # Extract Links
        link = box.find('a', class_='title')
        link_url = link['href'] if link else 'N/A'
        Links.append(link_url)


# -------------------------------
# 3. Main Execution
# -------------------------------

def main():
    job_types = {
        "Data Science": "https://www.naukri.com/data-scientist-data-science-jobs-{page_number}?k=data%20scientist%2C%20data%20science&nignbevent_src=jobsearchDeskGNB",
        "Machine Learning Engineering": "https://www.naukri.com/machine-learning-engineer-jobs-{page_number}?k=machine%20learning%20engineer&nignbevent_src=jobsearchDeskGNB",
        "AI Engineering": "https://www.naukri.com/ai-engineer-jobs-{page_number}?k=ai%20engineer&nignbevent_src=jobsearchDeskGNB",
        "Software Engineering": "https://www.naukri.com/software-engineering-jobs-{page_number}?k=software%20engineering&nignbevent_src=jobsearchDeskGNB",
        "Software Testing": "https://www.naukri.com/software-testing-jobs-{page_number}?k=software%20testing&nignbevent_src=jobsearchDeskGNB",
        "Cloud Engineering": "https://www.naukri.com/cloud-engineering-jobs-{page_number}?k=cloud%20engineering&nignbevent_src=jobsearchDeskGNB",
        "Java Developers": "https://www.naukri.com/java-developer-jobs-{page_number}?k=java+developer&nignbevent_src=jobsearchDeskGNB"
    }
    
    # Define the range of pages you want to scrape manually
    pages_to_scrape = range(1, 21)      
    
    # Step 1: Scrape job listings for each job type and page
    for job_type, url_pattern in job_types.items():
        logger.info(f"Starting to scrape {job_type} jobs.")
        for page in pages_to_scrape:
            scrape_job_listings(page, job_type, url_pattern)
    
    # Close the driver after scraping all pages
    driver.quit()
    logger.info("Completed web scraping and closed the browser.")
    
    # Step 2: Create a DataFrame from the scraped data
    job_data = {
        'CompanyName': CompanyName,
        'JobRole': JobTitle,
        'Experience': Experience,
        'Skills': Skills,
        'Links': Links
    }
    
    df = pd.DataFrame(job_data)
    logger.info("Created initial DataFrame.")
    print("Initial DataFrame:")
    print(df.head())  # Display the top rows of the DataFrame for reference
    
    # Step 3: Save the enriched DataFrame to a CSV file
    output_file = 'job_descriptions.csv'
    df.to_csv(output_file, index=False)
    logger.info(f"DataFrame saved to '{output_file}'.")

if __name__ == "__main__":
    main()


INFO:WDM:====== WebDriver manager ======
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:There is no [win64] chromedriver "146.0.7680.165" for browser google-chrome "146.0.7680" in cache
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:WebDriver version 146.0.7680.165 selected
INFO:WDM:Modern chrome version https://storage.googleapis.com/chrome-for-testing-public/146.0.7680.165/win32/chromedriver-win32.zip
INFO:WDM:About to download new driver from https://storage.googleapis.com/chrome-for-testing-public/146.0.7680.165/win32/chromedriver-win32.zip
INFO:WDM:Driver downloading response is 200
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Driver has been saved in cache [C:\Users\janum\.wdm\drivers\chromedriver\win64\146.0.7680.165]
INFO:root:Starting to scrape Data Science jobs.
INFO:root:Fetching URL: https://www.naukri.com/data-scientist-data-science-jobs-1?k=data%20sc

Initial DataFrame:
            CompanyName                                            JobRole  \
0      Greedygame Media        Founding Data Scientist / Data Science Lead   
1  Han Digital Solution               Data Scientist/data science - Gen AI   
2           Bristlecone                     Data Scientist - Data Science-   
3      Vidushi Infotech                                     Data Scientist   
4             Accenture  S&C Global Network - AI - Retail - Consultant ...   

  Experience                                             Skills  \
0  15-20 Yrs  SIDE, Trade, Usage, data science, Analytical, ...   
1    6-9 Yrs  algorithms, software design, machine learning ...   
2    1-6 Yrs                        data science, Science, Data   
3    3-8 Yrs  Data Science, Predictive Modeling, Pyspark, Lo...   
4    4-9 Yrs  python, natural language processing, machine l...   

                                               Links  
0  https://www.naukri.com/job-listings-founding-d...  

<br>
<br>
<br>

### **Analysis**

<br>

#### **1. Dataset**

In [25]:
jobs_df = pd.read_csv("job_descriptions.csv")
jobs_df.shape

(2789, 5)

In [26]:
jobs_df.head()

,CompanyName,JobRole,Experience,Skills,Links
0,Greedygame Media,Founding Data Scientist / Data Science Lead,15-20 Yrs,"SIDE, Trade, Usage, data science, Analytical, ...",https://www.naukri.com/job-listings-founding-d...
1,Han Digital Solution,Data Scientist/data science - Gen AI,6-9 Yrs,"algorithms, software design, machine learning ...",https://www.naukri.com/job-listings-data-scien...
2,Bristlecone,Data Scientist - Data Science-,1-6 Yrs,"data science, Science, Data",https://www.naukri.com/job-listings-data-scien...
3,Vidushi Infotech,Data Scientist,3-8 Yrs,"Data Science, Predictive Modeling, Pyspark, Lo...",https://www.naukri.com/job-listings-data-scien...
4,Accenture,S&C Global Network - AI - Retail - Consultant ...,4-9 Yrs,"python, natural language processing, machine l...",https://www.naukri.com/job-listings-s-c-global...


In [27]:
jobs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2789 entries, 0 to 2788
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   CompanyName  2789 non-null   object
 1   JobRole      2789 non-null   object
 2   Experience   2769 non-null   object
 3   Skills       2789 non-null   object
 4   Links        2789 non-null   object
dtypes: object(5)
memory usage: 109.1+ KB


In [28]:
jobs_df.duplicated().sum()

np.int64(139)

In [29]:
jobs_df.isna().sum()

CompanyName     0
JobRole         0
Experience     20
Skills          0
Links           0
dtype: int64

<br>

#### **2. Data Cleaning**

In [30]:
jobs_df.drop_duplicates(inplace = True) 
jobs_df.duplicated().sum()

np.int64(0)

In [31]:
jobs_df = jobs_df[ ~jobs_df['Experience'].isnull() ] 
jobs_df.isna().sum()

CompanyName    0
JobRole        0
Experience     0
Skills         0
Links          0
dtype: int64

In [32]:
jobs_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2632 entries, 0 to 2788
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   CompanyName  2632 non-null   object
 1   JobRole      2632 non-null   object
 2   Experience   2632 non-null   object
 3   Skills       2632 non-null   object
 4   Links        2632 non-null   object
dtypes: object(5)
memory usage: 123.4+ KB


In [33]:
jobs_df.shape

(2632, 5)

<br>

#### **3. Feature Engineering**

<br>

##### **i. Experience**

In [34]:
jobs_df['Experience'].unique()

array(['15-20 Yrs', '6-9 Yrs', '1-6 Yrs', '3-8 Yrs', '4-9 Yrs', '3-5 Yrs',
       '8-13 Yrs', '2-7 Yrs', '3-7 Yrs', '10-15 Yrs', '5-10 Yrs',
       '4-6 Yrs', '0-8 Yrs', '3-6 Yrs', '1-3 Yrs', '2-5 Yrs', '1-2 Yrs',
       '0-2 Yrs', '2-6 Yrs', '2-4 Yrs', '0-3 Yrs', '5-8 Yrs', '4-8 Yrs',
       '4-5 Yrs', '5-15 Yrs', '1-7 Yrs', '1-4 Yrs', '5-7 Yrs', '9-14 Yrs',
       '6-11 Yrs', '7-12 Yrs', '8-10 Yrs', '1-5 Yrs', '2-8 Yrs',
       '10-18 Yrs', '6-8 Yrs', '10-14 Yrs', '1-8 Yrs', '8-12 Yrs',
       '2-3 Yrs', '5-9 Yrs', '15-18 Yrs', '5-6 Yrs', '5-12 Yrs',
       '0-5 Yrs', '0-1 Yrs', '7-10 Yrs', '10-20 Yrs', '6-10 Yrs',
       '10-16 Yrs', '7-11 Yrs', '0-7 Yrs', '4-7 Yrs', '7-9 Yrs',
       '1-10 Yrs', '0-4 Yrs', '12-20 Yrs', '3-4 Yrs', '2-14 Yrs',
       '6-7 Yrs', '20-25 Yrs', '3-10 Yrs', '8-15 Yrs', '8-9 Yrs',
       '8-11 Yrs', '13-18 Yrs', '14-24 Yrs', '9-12 Yrs', '10-12 Yrs',
       '6-15 Yrs', '7-8 Yrs', '11-18 Yrs', '5-13 Yrs', '15-25 Yrs',
       '2-9 Yrs', '6-12 Yrs', '17-22 Yrs

In [35]:
jobs_df["Experience"] = jobs_df["Experience"].str.extract(r"(\d+)").astype(float)
jobs_df['Experience'].unique()

array([15.,  6.,  1.,  3.,  4.,  8.,  2., 10.,  5.,  0.,  9.,  7., 12.,
       20., 13., 14., 11., 17., 18., 16.])

In [36]:
jobs_df['Experience'].value_counts().sort_values(ascending=True) 

Experience
17.0      1
16.0      2
20.0      2
18.0      2
13.0      3
14.0      5
11.0      9
15.0     20
9.0      26
12.0     28
10.0     80
7.0     132
0.0     135
8.0     143
1.0     155
6.0     241
4.0     280
2.0     415
3.0     438
5.0     515
Name: count, dtype: int64

If I am interested in Saving them

In [60]:
zero_exp_jobs = jobs_df[jobs_df['Experience'] < 1]
zero_exp_jobs.head()

,CompanyName,JobRole,Experience,Skills,Links
16,Western Digital,"Scientist 5, Data Science",0.0,"Automation, Prototype, Publishing, Coding, Dis...",https://www.naukri.com/job-listings-scientist-...
37,Hbic Solutions,Data Scientist,0.0,"deep learning, data science, Artificial Intell...",https://www.naukri.com/job-listings-data-scien...
49,Biocube,Data Scientist,0.0,"algorithms, python, modeling, data analysis, d...",https://www.naukri.com/job-listings-data-scien...
105,Busigence Technologies,Data Engineer - Python,0.0,"deep learning, Data analysis, data science, Ma...",https://www.naukri.com/job-listings-data-engin...
115,Clustor Computing,Data Analyst,0.0,"Analytical skills, Data analysis, Manager Qual...",https://www.naukri.com/job-listings-data-analy...


In [61]:
zero_exp_jobs.to_csv('zero_experience_jobs.csv', index=False)

<br>

##### **ii. Skills**

In [62]:
jobs_df.head()

,CompanyName,JobRole,Experience,Skills,Links
0,Greedygame Media,Founding Data Scientist / Data Science Lead,15.0,"SIDE, Trade, Usage, data science, Analytical, ...",https://www.naukri.com/job-listings-founding-d...
1,Han Digital Solution,Data Scientist/data science - Gen AI,6.0,"algorithms, software design, machine learning ...",https://www.naukri.com/job-listings-data-scien...
2,Bristlecone,Data Scientist - Data Science-,1.0,"data science, Science, Data",https://www.naukri.com/job-listings-data-scien...
3,Vidushi Infotech,Data Scientist,3.0,"Data Science, Predictive Modeling, Pyspark, Lo...",https://www.naukri.com/job-listings-data-scien...
4,Accenture,S&C Global Network - AI - Retail - Consultant ...,4.0,"python, natural language processing, machine l...",https://www.naukri.com/job-listings-s-c-global...


In [63]:
jobs_df['Skills'][0]

'SIDE, Trade, Usage, data science, Analytical, Subject Matter Expert, Gaming, Statistics'

In [64]:
jobs_df["Skills"] = jobs_df["Skills"].str.replace(",", "", regex=False)
jobs_df['Skills'][0]

'SIDE Trade Usage data science Analytical Subject Matter Expert Gaming Statistics'

<br>

#### **4. Saving the File**

In [65]:
jobs_df.head()

,CompanyName,JobRole,Experience,Skills,Links
0,Greedygame Media,Founding Data Scientist / Data Science Lead,15.0,SIDE Trade Usage data science Analytical Subje...,https://www.naukri.com/job-listings-founding-d...
1,Han Digital Solution,Data Scientist/data science - Gen AI,6.0,algorithms software design machine learning fr...,https://www.naukri.com/job-listings-data-scien...
2,Bristlecone,Data Scientist - Data Science-,1.0,data science Science Data,https://www.naukri.com/job-listings-data-scien...
3,Vidushi Infotech,Data Scientist,3.0,Data Science Predictive Modeling Pyspark Logis...,https://www.naukri.com/job-listings-data-scien...
4,Accenture,S&C Global Network - AI - Retail - Consultant ...,4.0,python natural language processing machine lea...,https://www.naukri.com/job-listings-s-c-global...


In [66]:
jobs_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2632 entries, 0 to 2788
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CompanyName  2632 non-null   object 
 1   JobRole      2632 non-null   object 
 2   Experience   2632 non-null   float64
 3   Skills       2632 non-null   object 
 4   Links        2632 non-null   object 
dtypes: float64(1), object(4)
memory usage: 187.9+ KB


In [67]:
jobs_df.to_csv('cleaned_job_descriptions.csv', index=False)

In [68]:
jobs_df = pd.read_csv("cleaned_job_descriptions.csv")
jobs_df.shape

(2632, 5)